# RLT — actor-critic head walkthrough

Manual exploration of the RLT SAC head on a synthetic 'insertion' task. We model the residual-action setting:

- A frozen VLA emits a base action `a₀` that's already 90 % correct.
- The RLT head produces a small residual `Δa` ∈ [-0.05, +0.05] aligned with a hidden target.
- Reward = `-|action_total - target_action|`. RLT must learn to shift the action in the right direction *without* destabilizing.

Paper: `papers/2026-03-19_rlt_precise-manipulation-online-rl.pdf`

In [ ]:
import torch
import matplotlib.pyplot as plt

from pi_stack.rlt.rl_token import RLTConfig, RLTHead, RLTTrainer, ReplayBuffer

torch.manual_seed(0)
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', device)

## 1. Synthetic insertion env

State is `(target_offset, jitter)`. The VLA's base action is a noisy version of the target; the residual must close the gap. Episodes are 1 step long (terminal), so this is effectively contextual-bandit SAC — clean for showing the actor-critic mechanics.

In [ ]:
ACTION_DIM = 4
RL_TOKEN_DIM = 8
STATE_DIM = 4

def synth_step(batch_size: int):
    """Return (rl_token, state, base_action, target_action)."""
    target = 0.1 * torch.randn(batch_size, ACTION_DIM)        # tight insertion tolerance
    base   = target + 0.04 * torch.randn(batch_size, ACTION_DIM)  # frozen VLA is close
    state  = torch.cat([target, torch.zeros(batch_size, STATE_DIM - ACTION_DIM)], dim=-1)
    rl_token = torch.randn(batch_size, RL_TOKEN_DIM)
    return rl_token, state, base, target

def reward(action_total, target_action):
    return -((action_total - target_action) ** 2).sum(-1)

In [ ]:
head = RLTHead(RLTConfig(
    rl_token_dim=RL_TOKEN_DIM, state_dim=STATE_DIM, action_dim=ACTION_DIM,
    actor_hidden=64, critic_hidden=64,
    action_residual_scale=0.05,
    device=device,
))
trainer = RLTTrainer(head)
buf = ReplayBuffer(
    capacity=4096,
    rl_token_dim=RL_TOKEN_DIM, state_dim=STATE_DIM, action_dim=ACTION_DIM,
    device=device,
)
print('actor params  :', sum(p.numel() for p in head.actor.parameters()))
print('q1 params     :', sum(p.numel() for p in head.q1.parameters()))

## 2. Training loop

Collect transitions, push into replay, train SAC. Track reward over time.

In [ ]:
STEPS = 600
WARMUP = 100
BATCH = 64
history = {'reward_mean': [], 'q1_loss': [], 'actor_loss': [], 'alpha': []}

for step in range(STEPS):
    rl_tok, state, base, target = synth_step(batch_size=1)
    rl_tok, state = rl_tok.to(device), state.to(device)
    delta_a, _ = head.act(rl_tok, state)
    action_total = base.to(device) + delta_a.detach()
    r = reward(action_total, target.to(device))

    buf.add(
        rl_token=rl_tok[0], state=state[0],
        action=delta_a[0].detach(), reward=r[0].detach(),
        next_rl_token=rl_tok[0], next_state=state[0],
        done=True,
    )

    if step > WARMUP and len(buf) >= BATCH:
        metrics = trainer.update(buf.sample(BATCH))
        history['q1_loss'].append(metrics['q1_loss'])
        history['actor_loss'].append(metrics['actor_loss'])
        history['alpha'].append(metrics['alpha'])

    # Track *expected* reward at every step.
    history['reward_mean'].append(float(r.mean()))

print(f'final mean reward (last 100 steps): {torch.tensor(history["reward_mean"][-100:]).mean().item():.4f}')
print(f'initial mean reward (first 100 steps): {torch.tensor(history["reward_mean"][:100]).mean().item():.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
axes[0, 0].plot(history['reward_mean']); axes[0, 0].set(title='reward per step', xlabel='step')
axes[0, 1].plot(history['q1_loss']);    axes[0, 1].set(title='Q1 loss (post-warmup)', xlabel='update step')
axes[1, 0].plot(history['actor_loss']); axes[1, 0].set(title='actor loss', xlabel='update step')
axes[1, 1].plot(history['alpha']);      axes[1, 1].set(title='entropy α', xlabel='update step')
fig.tight_layout()

## 3. Evaluation: deterministic vs stochastic

At eval time the policy collapses to its mean (no exploration noise). The residual should remain bounded by `action_residual_scale`.

In [ ]:
rl_tok, state, base, target = synth_step(batch_size=256)
rl_tok, state = rl_tok.to(device), state.to(device)
with torch.no_grad():
    delta_det, _ = head.act(rl_tok, state, deterministic=True)
    delta_stoch, _ = head.act(rl_tok, state, deterministic=False)
print(f'deterministic Δa  range : [{delta_det.min().item():+.4f}, {delta_det.max().item():+.4f}]')
print(f'stochastic    Δa  range : [{delta_stoch.min().item():+.4f}, {delta_stoch.max().item():+.4f}]')
print(f'action_residual_scale   :  {head.config.action_residual_scale}')

## Takeaways

- The actor + twin critics + entropy-auto-tuned α all train cleanly on the toy task.
- Residual scale is enforced via the final tanh-squash + multiply — Δa stays within ±5 % of the base action.
- Replace `synth_step` with a real env wrapper (e.g. `pi_stack.envs.libero.LiberoEnv`) and `vla.extract_rl_token(obs)` once the VLA is wired in.